# Acquisition des données play-by-play de la LNH

Ce notebook présente la méthode utilisée pour récupérer les données
play-by-play de la LNH pour les saisons 2016-2017 à 2023-2024.

L'objectif est de construire un outil réutilisable permettant :

- de récupérer les identifiants des matchs d'une saison ;
- de télécharger les données play-by-play depuis l'API de la LNH ;
- de sauvegarder les données brutes localement ;
- d'utiliser une copie locale lorsqu'un match a déjà été téléchargé ;
- de séparer les données par saison et par type de match
  (saison régulière et séries éliminatoires).

Les fonctions principales sont implémentées dans le module
`ift6758.data_acquisition`.

## 1. Importation des fonctions d'acquisition

La logique principale d'acquisition est placée dans un module Python
réutilisable plutôt que directement dans le notebook.

In [ ]:
from pathlib import Path

import pandas as pd

from ift6758.data_acquisition import (
    build_regular_game_id,
    download_game,
    download_season,
    get_game_ids,
)

## 2. Structure du GAME_ID

Chaque match de la LNH possède un identifiant unique (`GAME_ID`).

Pour un match de saison régulière, l'identifiant suit la structure :

`YYYY02NNNN`

où :

- `YYYY` correspond à l'année de début de la saison ;
- `02` correspond à un match de saison régulière ;
- `NNNN` correspond au numéro du match.

Par exemple, le premier match de saison régulière de la saison
2023-2024 possède l'identifiant `2023020001`.

In [ ]:
game_id = build_regular_game_id(2023, 1)

print("GAME_ID :", game_id)

## 3. Récupération des GAME_ID d'une saison

Plutôt que de générer tous les identifiants possibles, nous interrogeons
l'API de la LNH afin d'obtenir uniquement les matchs réellement associés
à une saison et à un type de match donné.

La fonction `get_game_ids` accepte :

- l'identifiant de la saison, par exemple `20232024` ;
- le type de match :
  - `2` pour la saison régulière ;
  - `3` pour les séries éliminatoires.

Cela permet d'éviter des requêtes inutiles vers des GAME_ID inexistants.

In [ ]:
regular_ids = get_game_ids(20232024, 2)
playoff_ids = get_game_ids(20232024, 3)

print("Saison régulière :", len(regular_ids))
print("Premier GAME_ID :", regular_ids[0])
print("Dernier GAME_ID :", regular_ids[-1])

print()

print("Séries éliminatoires :", len(playoff_ids))
print("Premier GAME_ID :", playoff_ids[0])
print("Dernier GAME_ID :", playoff_ids[-1])

## 4. Téléchargement d'un match et mise en cache locale

La fonction `download_game` récupère les données play-by-play d'un match
à partir de son `GAME_ID`.

Le comportement est le suivant :

1. si le fichier JSON existe déjà localement, les données sont chargées
   depuis le cache ;
2. sinon, une requête est envoyée à l'API de la LNH ;
3. la réponse JSON est sauvegardée localement ;
4. les données sont retournées sous forme de dictionnaire Python.

Cette stratégie permet d'éviter de télécharger plusieurs fois les mêmes
données et réduit le nombre de requêtes envoyées à l'API publique.

In [ ]:
game = download_game(
    2023020001,
    data_dir=Path("../data/raw/20232024/regular")
)

print("Game ID :", game["id"])
print("Date :", game["gameDate"])
print("Nombre d'événements :", len(game["plays"]))

Le message `Chargement depuis le cache` confirme qu'aucune nouvelle
requête HTTP n'a été envoyée lorsque les données étaient déjà disponibles
localement.

## 5. Téléchargement d'une saison complète

La fonction `download_season` combine la récupération des GAME_ID
et le téléchargement des données play-by-play.

Pour chaque saison, les matchs sont séparés en deux catégories :

- `regular` pour la saison régulière ;
- `playoffs` pour les séries éliminatoires.

Les fichiers JSON sont sauvegardés selon la structure suivante :

`data/raw/<saison>/<type_de_match>/`

Par exemple :

`data/raw/20232024/regular/`

et

`data/raw/20232024/playoffs/`

La fonction accepte également le paramètre optionnel `max_games`, utile
pour effectuer de petits tests avant de lancer un téléchargement complet.

In [ ]:
# Exemple de test avec deux matchs par type

download_season(20232024, max_games=2)

Pour télécharger une saison complète, il suffit d'omettre le paramètre
`max_games` :

```python
download_season(20232024)
```

Dans le projet, cette méthode a été utilisée pour récupérer les saisons
2016-2017 à 2023-2024.


In [ ]:
seasons = [20162017, 20172018, 20182019, 20192020,
           20202021, 20212022, 20222023, 20232024]

for season_id in seasons:
    download_season(season_id)

## 6. Validation des données téléchargées

Après le téléchargement, nous vérifions que les fichiers disponibles
localement correspondent aux matchs retournés par l'API.

Pour chaque saison, nous comparons :

- le nombre de GAME_ID retournés par l'API ;
- le nombre de fichiers JSON sauvegardés localement.

Pour les séries éliminatoires, certains GAME_ID retournés par la liste
des matchs ne possèdent pas de données play-by-play disponibles.
Dans ce cas, l'endpoint play-by-play retourne une erreur HTTP 404.
Ces matchs sont ignorés par l'outil plutôt que de créer des fichiers
JSON artificiels.

In [ ]:
validation_results = []

seasons = [
    20162017,
    20172018,
    20182019,
    20192020,
    20202021,
    20212022,
    20222023,
    20232024,
]

data_root = Path("../data/raw")

for season_id in seasons:
    expected_regular = len(get_game_ids(season_id, 2))
    expected_playoffs = len(get_game_ids(season_id, 3))

    regular_dir = data_root / str(season_id) / "regular"
    playoffs_dir = data_root / str(season_id) / "playoffs"

    local_regular = len(list(regular_dir.glob("*.json")))
    local_playoffs = len(list(playoffs_dir.glob("*.json")))

    validation_results.append(
        {
            "Saison": season_id,
            "Regular local": local_regular,
            "Regular API": expected_regular,
            "Playoffs local": local_playoffs,
            "Playoffs API": expected_playoffs,
            "Sans PBP": expected_playoffs - local_playoffs,
        }
    )

validation_df = pd.DataFrame(validation_results)

In [ ]:
display(validation_df)

Les saisons régulières sont entièrement disponibles localement.

Pour certaines saisons, le nombre de fichiers de séries éliminatoires
est inférieur au nombre de GAME_ID retournés par l'API. Cette différence
correspond aux GAME_ID pour lesquels l'endpoint play-by-play retourne
une réponse HTTP 404.

L'outil traite ces cas comme des données play-by-play indisponibles et
poursuit le téléchargement des autres matchs.

## 7. Résumé

L'outil d'acquisition permet donc :

- d'obtenir les GAME_ID d'une saison ;
- de télécharger les données play-by-play d'un match ;
- de mettre automatiquement les réponses en cache localement ;
- de télécharger une saison complète ;
- de séparer les données par saison régulière et séries éliminatoires ;
- de continuer le téléchargement lorsqu'un GAME_ID ne possède pas de données play-by-play accessibles.

Les données brutes des saisons 2016-2017 à 2023-2024 ont été récupérées
et sont conservées localement dans `data/raw/`.

Le dossier `data/` est exclu du dépôt Git afin d'éviter de versionner
un grand nombre de fichiers de données brutes.